# Hybrid Skill Gap Analysis

This notebook builds a reusable matcher from the actual `skills`, `Job Title`, and `Role` fields in the job-posting dataset. It applies exact matching first, then RapidFuzz, then optional Sentence Transformer similarity. The transformer is not serialized into the joblib artifact.

## Load and inspect the job corpus

The 50,000-row CSV is read with pandas' CSV parser. Contact fields are not used.

In [ ]:
from pathlib import Path
import re
import unicodedata
import joblib
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process

PROJECT_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'ml' / 'data').exists())
DATA_PATH = PROJECT_ROOT / 'ml' / 'data' / 'job_descriptions.csv'
MODEL_DIR = PROJECT_ROOT / 'ml' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), DATA_PATH
df = pd.read_csv(DATA_PATH, low_memory=False)
print('shape:', df.shape)
print('columns:', df.columns.tolist())
display(df[['Job Title', 'Role', 'skills']].head())
print('missing values:', df.isna().sum().loc[lambda values: values.gt(0)].to_dict())
print('duplicate rows:', int(df.duplicated().sum()))
print('unique job titles:', int(df['Job Title'].nunique()))

## Normalize and extract observed skills

The source uses inconsistent separators and occasionally contains encoding artifacts. A conservative vocabulary is built from terms that actually occur in the `skills` corpus. Safe aliases are applied only when their canonical term is observed.

In [ ]:
def normalize_text(value):
    text = '' if pd.isna(value) else str(value)
    text = text.replace('â€™', "'").replace('â€“', '-')
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_skill(value):
    return re.sub(r'[^a-z0-9+#.]+', ' ', normalize_text(value)).strip()

safe_aliases = {
    'ml': 'machine learning', 'machine-learning': 'machine learning',
    'ai': 'artificial intelligence', 'powerbi': 'power bi',
    'ms excel': 'excel', 'microsoft excel': 'excel',
    'js': 'javascript', 'postgresql': 'postgres',
}
candidate_terms = [
    'machine learning', 'artificial intelligence', 'deep learning', 'data analysis', 'data visualization',
    'project management', 'problem solving', 'communication skills', 'time management', 'critical thinking',
    'social media', 'content creation', 'customer service', 'financial analysis', 'business analysis',
    'statistical analysis', 'database management', 'data management', 'digital marketing', 'market research',
    'python', 'sql', 'excel', 'tableau', 'power bi', 'r programming', 'java', 'javascript', 'typescript',
    'html', 'css', 'react', 'angular', 'node.js', 'c++', 'c#', 'php', 'ruby', 'go', 'matlab', 'sas',
    'tensorflow', 'pytorch', 'scikit learn', 'spark', 'hadoop', 'aws', 'azure', 'docker', 'git', 'linux',
    'autocad', 'accounting', 'sales', 'recruitment', 'seo', 'advertising', 'copywriting',
]
corpus_text = ' '.join(df['skills'].map(normalize_text))
observed_terms = []
for term in sorted(candidate_terms, key=len, reverse=True):
    if re.search(r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])', corpus_text):
        observed_terms.append(term)
observed_terms = sorted(set(observed_terms), key=lambda value: (len(value), value))
aliases = {key: value for key, value in safe_aliases.items() if value in observed_terms}
print('observed canonical terms:', len(observed_terms))
print(observed_terms)

In [ ]:
def extract_skills(value):
    text = normalize_skill(value)
    found = []
    for term in sorted(observed_terms, key=len, reverse=True):
        if re.search(r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])', text):
            found.append(term)
    if found:
        return sorted(set(found))
    fragments = re.split(r'[,;|/]+', text)
    return sorted({fragment.strip() for fragment in fragments if len(fragment.strip()) > 1})

def normalize_skill_list(values):
    if isinstance(values, str):
        values = re.split(r'[,;|/]+', values)
    normalized = []
    for value in values or []:
        item = normalize_skill(value)
        item = aliases.get(item, item)
        if item:
            normalized.append(item)
    return sorted(set(normalized))

df['normalized_title'] = df['Job Title'].map(normalize_text)
df['normalized_role'] = df['Role'].map(normalize_text)
df['extracted_skills'] = df['skills'].map(extract_skills)
title_to_display = df.drop_duplicates('normalized_title').set_index('normalized_title')['Job Title'].to_dict()
CORE_MIN_FREQUENCY = 0.50
OPTIONAL_MIN_FREQUENCY = 0.20

title_skill_profiles = {}
for title, title_rows in df.groupby('normalized_title', sort=True):
    non_empty = title_rows[title_rows['extracted_skills'].map(bool)]
    posting_count = len(non_empty)
    counts = pd.Series(dtype='int64')
    if posting_count:
        counts = pd.Series([skill for skills in non_empty['extracted_skills'] for skill in set(skills)]).value_counts()
    records = []
    for skill, count in counts.items():
        frequency = float(count / posting_count)
        category = 'core' if frequency >= CORE_MIN_FREQUENCY else 'optional' if frequency >= OPTIONAL_MIN_FREQUENCY else 'rare'
        records.append({'skill': skill, 'posting_count': int(count), 'frequency': frequency, 'category': category})
    records.sort(key=lambda record: (-record['frequency'], record['skill']))
    title_skill_profiles[title] = {'posting_count': int(posting_count), 'skills': records}
requirements_by_title = {
    title: [record['skill'] for record in profile['skills']]
    for title, profile in title_skill_profiles.items()
}
core_skills_by_title = {
    title: [record['skill'] for record in profile['skills'] if record['category'] == 'core']
    for title, profile in title_skill_profiles.items()
}
optional_skills_by_title = {
    title: [record['skill'] for record in profile['skills'] if record['category'] == 'optional']
    for title, profile in title_skill_profiles.items()
}
rare_skills_by_title = {
    title: [record['skill'] for record in profile['skills'] if record['category'] == 'rare']
    for title, profile in title_skill_profiles.items()
}
print('titles with requirements:', len(requirements_by_title))
print('core/optional/rare counts:', sum(map(len, core_skills_by_title.values())), sum(map(len, optional_skills_by_title.values())), sum(map(len, rare_skills_by_title.values())))
display(df[['Job Title', 'skills', 'extracted_skills']].head(3))

## Matching pipeline and controlled evaluation

Exact matches are authoritative. Fuzzy matches are accepted only above a strong threshold. Semantic matching is attempted only for remaining skills and only when the optional pretrained model is available. Each candidate skill can be assigned to at most one requirement. The evaluation below uses manually defined controlled cases for aliases, spelling variations, unrelated skills, and semantic relationships; it is not a real-world benchmark.

In [ ]:
FUZZY_THRESHOLD = 90
SEMANTIC_THRESHOLD = 0.72
MODEL_NAME = 'all-MiniLM-L6-v2'
semantic_model = None
semantic_status = 'unavailable'
try:
    from sentence_transformers import SentenceTransformer
    semantic_model = SentenceTransformer(MODEL_NAME)
    semantic_status = 'loaded'
except Exception as error:
    print('Semantic model unavailable; using exact/fuzzy fallback:', type(error).__name__)
print('semantic status:', semantic_status)

In [ ]:
def hybrid_match(candidate_skills, required_skills, fuzzy_threshold=FUZZY_THRESHOLD, semantic_threshold=SEMANTIC_THRESHOLD):
    candidates = normalize_skill_list(candidate_skills)
    required = normalize_skill_list(required_skills)
    matched = {}
    unused_candidates = set(candidates)

    for requirement in required:
        if requirement in unused_candidates:
            matched[requirement] = {'candidate': requirement, 'method': 'exact', 'score': 100.0}
            unused_candidates.remove(requirement)

    for requirement in required:
        if requirement in matched or not unused_candidates:
            continue
        choices = sorted(unused_candidates)
        best = process.extractOne(requirement, choices, scorer=fuzz.token_set_ratio)
        if best and best[1] >= fuzzy_threshold:
            matched[requirement] = {'candidate': best[0], 'method': 'fuzzy', 'score': float(best[1])}
            unused_candidates.remove(best[0])

    if semantic_model is not None and unused_candidates:
        unmatched_requirements = [item for item in required if item not in matched]
        candidate_list = sorted(unused_candidates)
        if unmatched_requirements:
            requirement_vectors = semantic_model.encode(unmatched_requirements, normalize_embeddings=True)
            candidate_vectors = semantic_model.encode(candidate_list, normalize_embeddings=True)
            scores = requirement_vectors @ candidate_vectors.T
            semantic_pairs = sorted(
                [(float(scores[row_index, candidate_index]), requirement, candidate_list[candidate_index])
                 for row_index, requirement in enumerate(unmatched_requirements)
                 for candidate_index in range(len(candidate_list))],
                key=lambda pair: (-pair[0], pair[1], pair[2]),
            )
            used_requirements = set(matched)
            used_candidates = set()
            for score, requirement, candidate in semantic_pairs:
                if score < semantic_threshold or requirement in used_requirements or candidate in used_candidates:
                    continue
                matched[requirement] = {'candidate': candidate, 'method': 'semantic', 'score': score}
                used_requirements.add(requirement)
                used_candidates.add(candidate)

    matched_list = [requirement for requirement in required if requirement in matched]
    missing_list = [requirement for requirement in required if requirement not in matched]
    return matched_list, missing_list, matched

def analyze_skill_gap(candidate_skills, target_job):
    title = normalize_text(target_job)
    profile = title_skill_profiles.get(title)
    if profile is None:
        return {
            'target_job': target_job, 'job_found': False, 'core_skills': [], 'optional_skills': [],
            'matched_skills': [], 'matched_core_skills': [], 'missing_skills': [], 'missing_core_skills': [],
            'skill_match_percentage': 0.0, 'core_skill_match_percentage': 0.0, 'recommended_skills': [],
        }
    core = core_skills_by_title[title]
    optional = optional_skills_by_title[title]
    rare = rare_skills_by_title[title]
    required = core + optional + rare
    matched, missing, details = hybrid_match(candidate_skills, required)
    matched_set = set(matched)
    missing_core = [skill for skill in core if skill not in matched_set]
    missing_optional = [skill for skill in optional if skill not in matched_set]
    percentage = round(100 * len(matched) / len(required), 2) if required else 0.0
    core_percentage = round(100 * (len(core) - len(missing_core)) / len(core), 2) if core else 100.0
    return {
        'target_job': title_to_display.get(title, target_job), 'job_found': True,
        'core_skills': core, 'optional_skills': optional, 'matched_skills': matched,
        'matched_core_skills': [skill for skill in core if skill in matched_set],
        'missing_skills': missing, 'missing_core_skills': missing_core,
        'skill_match_percentage': percentage, 'core_skill_match_percentage': core_percentage,
        'recommended_skills': missing_core + missing_optional, 'match_details': details,
    }

known_title = next(title for title, profile in title_skill_profiles.items() if profile['skills'])
known_requirements = requirements_by_title[known_title]
known_result = analyze_skill_gap(known_requirements[:2], known_title)
unknown_result = analyze_skill_gap(['python'], 'Job Title That Does Not Exist')
print(known_result)
print(unknown_result)

In [ ]:
# Controlled matcher sanity checks: manually labeled demonstrations, not a real-world benchmark.
controlled_cases = [
    {'name': 'exact match', 'candidate': ['Python'], 'required': ['Python'], 'expected_match': True, 'expected_method_category': 'exact'},
    {'name': 'machine learning alias', 'candidate': ['ML'], 'required': ['Machine Learning'], 'expected_match': True, 'expected_method_category': 'alias'},
    {'name': 'PowerBI alias', 'candidate': ['PowerBI'], 'required': ['Power BI'], 'expected_match': True, 'expected_method_category': 'alias'},
    {'name': 'minor spelling variation', 'candidate': ['Pythn'], 'required': ['Python'], 'expected_match': True, 'expected_method_category': 'fuzzy'},
    {'name': 'unrelated skill', 'candidate': ['Python'], 'required': ['Java'], 'expected_match': False, 'expected_method_category': 'none'},
    {'name': 'semantic related skill', 'candidate': ['Deep Learning'], 'required': ['Machine Learning'], 'expected_match': True, 'expected_method_category': 'semantic'},
]

def evaluate_controlled_cases(cases, fuzzy_threshold, semantic_threshold):
    rows = []
    for case in cases:
        matched, missing, details = hybrid_match(case['candidate'], case['required'], fuzzy_threshold, semantic_threshold)
        predicted_match = bool(matched)
        predicted_method = details[matched[0]]['method'] if matched else 'none'
        method_matches = (
            case['expected_method_category'] == 'alias' and predicted_method == 'exact'
        ) or predicted_method == case['expected_method_category']
        rows.append({
            'case': case['name'],
            'expected_match': case['expected_match'],
            'predicted_match': predicted_match,
            'expected_method_category': case['expected_method_category'],
            'predicted_method': predicted_method,
            'passed': case['expected_match'] == predicted_match and method_matches,
        })
    return pd.DataFrame(rows)

controlled_results = evaluate_controlled_cases(controlled_cases, FUZZY_THRESHOLD, SEMANTIC_THRESHOLD)
print('Controlled matcher sanity checks')
display(controlled_results)
print('These manually labeled demonstrations are sanity checks only, not a real-world benchmark or production accuracy estimate.')

## Save configuration artifact

The artifact stores normalized requirements, safe aliases, thresholds, and the transformer model name. The transformer weights remain external and can be loaded by a future engine when available.

In [ ]:
artifact = {
    'artifact_type': 'hybrid_skill_gap_configuration',
    'version': 2,
    'source_file': DATA_PATH.name,
    'requirements_by_title': requirements_by_title,
    'title_to_display': title_to_display,
    'title_skill_profiles': title_skill_profiles,
    'skill_frequency_data': title_skill_profiles,
    'core_skills_by_title': core_skills_by_title,
    'optional_skills_by_title': optional_skills_by_title,
    'rare_skills_by_title': rare_skills_by_title,
    'core_min_frequency': CORE_MIN_FREQUENCY,
    'optional_min_frequency': OPTIONAL_MIN_FREQUENCY,
    'observed_terms': observed_terms,
    'aliases': aliases,
    'fuzzy_threshold': FUZZY_THRESHOLD,
    'semantic_threshold': SEMANTIC_THRESHOLD,
    'semantic_model_name': MODEL_NAME,
    'semantic_status_at_build': semantic_status,
    'matching_order': ['exact', 'fuzzy', 'semantic'],
    'one_to_one_matching': True,
    'fuzzy_scorer': 'rapidfuzz.fuzz.token_set_ratio',
    'controlled_evaluation': {
        'name': 'Controlled matcher sanity checks',
        'case_count': len(controlled_cases),
        'results': controlled_results.to_dict('records'),
        'is_real_world_benchmark': False,
    },
    'output_schema': ['target_job', 'job_found', 'core_skills', 'optional_skills', 'matched_skills', 'matched_core_skills', 'missing_skills', 'missing_core_skills', 'skill_match_percentage', 'core_skill_match_percentage', 'recommended_skills'],
    'sensitive_columns_excluded': ['Contact Person', 'Contact'],
}
artifact_path = MODEL_DIR / 'skill_gap_model.joblib'
joblib.dump(artifact, artifact_path)
assert artifact_path.exists() and artifact_path.stat().st_size > 0
loaded_artifact = joblib.load(artifact_path)
assert loaded_artifact['artifact_type'] == 'hybrid_skill_gap_configuration'
assert loaded_artifact['skill_frequency_data']
assert loaded_artifact['one_to_one_matching'] is True
print('saved:', artifact_path)
print('titles saved:', len(loaded_artifact['requirements_by_title']))